# 04 · Entrenamiento: recursos y piloto

Primero medimos si el batch cabe en la GPU; después ejecutamos un piloto con **50 épocas propuestas**, tomadas del YAML. Este notebook usa las mismas funciones de entrenamiento que podrán llamar los trials de Optuna. No inicia búsquedas ni evaluación final.

Requiere el pipeline del 02 y V01–V14 aprobadas. Tras revisar un reporte completo de 03, registrar `verification.status: approved` y la ruta relativa a TT_ECoG de su `approved_pipeline_manifest.json` en `verification.approved_manifest_file`. Esa selección documenta la aprobación humana; el candidato por sí solo no la concede. La purga debe estar decidida.

La prueba de recursos y el piloto se ejecutan sobre el sujeto con más canales y el primer fold de etapa 1. Es una primera medición, no evidencia de convergencia para todos los sujetos y learning rates. El máximo final sigue pendiente.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

In [ ]:
env_check = subprocess.run([sys.executable, str(PROJECT / 'scripts/check_environment.py'), '--install-missing'],
                           capture_output=True, text=True)
print(env_check.stdout + env_check.stderr)  # Colab no muestra la salida de subprocesos sin captura.
env_check.check_returncode()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest>=8,<10', 'nbformat>=5,<6'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
import torch
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '| torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
CONFIG = PROJECT / 'configs/experiment_config.yaml'


## Configuración y GPU

In [ ]:
import math
from time import perf_counter
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tt_ecog.config import load_config
from tt_ecog.datasets import load_fold
from tt_ecog.io import approved_pipeline, save_training_checkpoint
from tt_ecog.model import build_model, reference_architecture
from tt_ecog.losses import make_loss
from tt_ecog.training import (device_info, fold_seed, seed_run, make_loaders, make_optimizer,
    representative_architectures, profile_architecture, train_epoch, evaluate,
    synchronize, reset_peak_memory, peak_memory)
from tt_ecog.reporting import (start_training_run, record_training_epoch,
    finish_training_run)

config = load_config(CONFIG)

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Activa un runtime con GPU en Colab para medir recursos y entrenar.')
device = torch.device('cuda')
print(device_info(device))

## Datos preparados y aprobación del 03
Se lee el pipeline seleccionado, sin consultar el EDA ni las etiquetas de test. Los escaladores ya están ajustados por fold.

In [ ]:
pipeline_dir = approved_pipeline(config, DRIVE_ROOT)
subject = max(config.data['project']['subjects'], key=lambda s:
    config.data['raw_data']['expected_channels_nominal'][s] - len(config.excluded_channels(s)))
val_block = config.data['validation']['stage1_validation_blocks'][0]
fold = load_fold(config, pipeline_dir / 'folds', subject, val_block, verify=False)
print('pipeline_id:', fold.pipeline_id)
print(subject, fold.name, '| train:', len(fold.train_windows), '| val:', len(fold.val_windows))

## Preparar la corrida
El batch sigue siendo candidato hasta revisar la medición. Las épocas propuestas se usan solo para este piloto, sin rellenar `search_epochs` ni `final_max_epochs`.

In [ ]:
settings = config.data['training']
batch_size = settings['batch_size'] if settings['batch_size'] is not None else settings['batch_size_initial_candidate']
epochs = settings['search_epochs_candidate']
seed = fold_seed(config, fold.subject, fold.val_block)
arch = reference_architecture(config)
parameters = {'purpose': 'pilot', 'architecture': arch.key, 'seed': seed,
    'batch_size': batch_size, 'epochs': epochs, 'lr': settings['reference_learning_rate'],
    'weight_decay': settings['reference_weight_decay']}
output, parameters = start_training_run(config, DRIVE_ROOT, fold, parameters, device)
print('Resultados:', output)
print('Piloto:', epochs, 'épocas | batch:', batch_size)

## Comprobación breve de recursos
Para cada arquitectura se crea un modelo nuevo: dos actualizaciones de calentamiento y tres medidas. Se mide forward, backward y AdamW, incluyendo sus estados en memoria. Se desechan esos pesos.

Si aparece `OOM`, reduce `training.batch_size_initial_candidate` y repite la corrida completa. No se cambia el batch automáticamente. Los tiempos por época son estimaciones de entrenamiento; la duración completa se medirá en el piloto. Esta prueba representativa no certifica las 27 arquitecturas.

In [ ]:
profiles = []
try:
    for candidate in representative_architectures(config):
        row = profile_architecture(config, fold, candidate, batch_size, device, seed)
        row['run_id'] = parameters['run_id']
        profiles.append(row)
        pd.DataFrame(profiles).to_csv(output / 'resource_profile.csv', index=False)
        display(pd.DataFrame([row]))
    if any(row['status'] != 'OK' for row in profiles):
        raise RuntimeError('El batch no cabe en todas las arquitecturas probadas; reducir el candidato y repetir.')
except (Exception, KeyboardInterrupt) as exc:
    finish_training_run(output, parameters, 'FAILED', 0, details=str(exc))
    raise
print('La memoria permite probar este batch; aún falta medir una época completa.')

## Modelo nuevo y optimizador
El piloto parte de una inicialización nueva, no de los pesos usados al medir memoria. AdamW a todos los parámetros entrenables, sin scheduler.

In [ ]:
seed_run(seed)
train_loader, val_loader = make_loaders(fold, batch_size, seed)
model = build_model(config, fold.blocks[fold.val_block].x.shape[0], arch).to(device)
optimizer = make_optimizer(model, parameters['lr'], parameters['weight_decay'])
loss_fn = make_loss(config)
print('Inicialización:', seed, '| épocas:', epochs, '| selección: última época')

## Entrenar y validar, época por época
Ejecuta esta celda después de revisar los recursos. Cada época recorre todas las ventanas de entrenamiento y reconstruye la trayectoria de validación mediante overlap-add. Se guardan loss, Pearson por dedo y macro, tiempo y memoria. Un Pearson indefinido o un fallo numérico detiene la corrida con diagnóstico.

Se conserva `checkpoint/last.pt`. No hay early stopping ni selección del mejor pico de validación. El checkpoint incluye pesos, AdamW y aleatoriedad de PyTorch/DataLoader para recuperar la última época completa en el mismo entorno.

In [ ]:
history = []
completed = 0
last_score = None
try:
    for epoch in range(1, epochs + 1):
        reset_peak_memory(device)
        synchronize(device)
        started = perf_counter()
        train_loss = train_epoch(model, optimizer, loss_fn, train_loader, device)
        metrics = evaluate(model, loss_fn, fold, val_loader, device,
                           config.data['metrics']['degenerate_std_relative'])
        synchronize(device)
        seconds = perf_counter() - started
        save_training_checkpoint(output / 'checkpoint/last.pt', model, optimizer, epoch, parameters, train_loader)
        row = record_training_epoch(output, parameters, epoch, train_loss, metrics, seconds, peak_memory(device))
        history.append(row)
        completed, last_score = epoch, metrics['val_pearson_full']
        print(f"Época {epoch}/{epochs} | train {train_loss:.5f} | val {metrics['val_loss']:.5f} | "
              f"Pearson {last_score:.4f} | {seconds:.1f} s")
except (Exception, KeyboardInterrupt) as exc:
    finish_training_run(output, parameters, 'FAILED', completed, last_score, details=str(exc))
    raise

## Curvas y decisión posterior
Revisar si 50 épocas resultan suficientes y el costo medido. Un piloto aislado no aprueba el presupuesto de búsqueda. Si se decide usarlo, documentar la decisión y fijar `training.search_epochs`; el máximo final se decide por separado.

In [ ]:
frame = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), layout='constrained')
frame.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0])
frame.plot(x='epoch', y='val_pearson_full', ax=axes[1])
frame.plot(x='epoch', y='elapsed_seconds', ax=axes[2])
axes[0].set_ylabel('Loss')
axes[1].set_ylabel('Pearson reconstruido')
axes[2].set_ylabel('Segundos de train + validación')
fig.savefig(output / 'learning_curves.png', dpi=config.data['eda']['figure_dpi'])
plt.show()
summary = finish_training_run(output, parameters, 'COMPLETE', completed, last_score)
print('Última época:', completed, '| Pearson:', last_score)
print('Duración medida (train + validación):', frame['elapsed_seconds'].sum() / 60, 'min')
print('No se aprobaron épocas ni batch automáticamente. Optuna no se ha ejecutado.')